# GPU-Accelerated AQI Prediction with Genetic Algorithm Optimisation
> CUDA-powered RMSE kernel + GA-tuned RandomForest on real-world AQI data

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from numba import cuda

## 1. Reproducibility

In [ ]:
np.random.seed(42)

## 2. Generate / Load Dataset

In [ ]:
# Load the real AQI dataset from disk
data = pd.read_csv("../data/dataset.csv")
data.head()

# Drop columns that are non-numeric or irrelevant to regression
drop_cols = ["City", "Datetime", "AQI_Bucket"]
data = data.drop(columns=[col for col in drop_cols if col in data.columns])

# Remove rows with missing sensor readings
data = data.dropna()

print("Cleaned Shape:", data.shape)

## 3. Preprocessing

In [ ]:
features = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

X = data[features].values
y = data["AQI"].values
y = y.clip(0, 500)          # cap AQI to valid EPA range

# Standardise features to zero-mean / unit-variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Stratified 80/20 split — random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

## 4. CUDA RMSE Kernel (GPU)

In [ ]:
@cuda.jit
def compute_rmse_gpu(y_true, y_pred, errors):
    """
    CUDA kernel — each thread computes the squared error for one sample.
    Results are stored in `errors` and reduced on the host.
    """
    idx = cuda.grid(1)
    if idx < y_true.size:
        diff = y_true[idx] - y_pred[idx]
        errors[idx] = diff * diff


def gpu_rmse(y_true, y_pred):
    """
    Host-side wrapper:
      1. Copies arrays to GPU device memory.
      2. Launches the CUDA kernel with 256 threads/block.
      3. Copies squared errors back to host and returns sqrt(mean).
    Falls back to sklearn's CPU implementation if no GPU is detected.
    """
    # NEW - actually tests the device
    try:
        cuda.select_device(0)
        gpu_available = True
    except:
        gpu_available = False

    if not gpu_available:
        raise Exception("CUDA not available")

    n = len(y_true)
    errors = np.zeros(n, dtype=np.float32)

    # Allocate and transfer data to device
    d_y_true = cuda.to_device(y_true.astype(np.float32))
    d_y_pred = cuda.to_device(y_pred.astype(np.float32))
    d_errors = cuda.to_device(errors)

    # Grid/block configuration: 256 threads per block
    threads = 256
    blocks = (n + threads - 1) // threads

    compute_rmse_gpu[blocks, threads](d_y_true, d_y_pred, d_errors)

    # Copy results back and compute final RMSE on host
    errors = d_errors.copy_to_host()
    return np.sqrt(np.mean(errors))

## 5. GPU Warm-Up

In [ ]:
_ = gpu_rmse(y_test[:100], np.zeros(100, dtype=np.float32))
print("GPU warmed up.") 

## 6. CPU vs GPU RMSE Benchmark

In [ ]:
# Train a default RF just to get a set of predictions to benchmark against
model = RandomForestRegressor(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_test)

# ── CPU timing ───────────────────────────────────────────────────────────────
start = time.perf_counter()
cpu_rmse_val = np.sqrt(mean_squared_error(y_test, preds))
cpu_time = time.perf_counter() - start

# ── GPU timing ───────────────────────────────────────────────────────────────
start = time.perf_counter()
gpu_rmse_val = gpu_rmse(y_test, preds)
gpu_time = time.perf_counter() - start

print(f"CPU RMSE : {cpu_rmse_val:.4f}  |  Time: {cpu_time*1e6:.2f} µs")
print(f"GPU RMSE : {gpu_rmse_val:.4f}  |  Time: {gpu_time*1e6:.2f} µs")
print(f"Speedup  : {cpu_time / gpu_time:.1f}×  (GPU vs CPU)")

## 7. Genetic Algorithm (GPU-accelerated fitness)

In [ ]:
class GAOptimizer:
    """
    Population-based Genetic Algorithm that searches RandomForest
    hyper-parameters (n_estimators, max_depth).

    Fitness function uses gpu_rmse so squared-error computation
    is offloaded to the CUDA kernel on each generation.
    """

    def __init__(self, X, y, pop_size=15, generations=15):
        self.X = X
        self.y = y
        self.pop_size = pop_size
        self.generations = generations
        self.history = []       # tracks best RMSE per generation

    # ── Population initialisation ─────────────────────────────────────────
    def initialize(self):
        """Randomly sample hyper-parameter combinations as the seed population."""
        return [
            {
                "n_estimators": int(np.random.randint(10, 100)),
                "max_depth":    int(np.random.randint(3, 15))
            }
            for _ in range(self.pop_size)
        ]

    # ── Fitness evaluation ────────────────────────────────────────────────
    def fitness(self, individual):
        """
        Split data, fit a RF with this individual's params,
        and return GPU RMSE on the validation fold.
        n_jobs=-1 uses all CPU cores for tree building.
        """
        X_tr, X_val, y_tr, y_val = train_test_split(
            self.X, self.y, test_size=0.2
        )
        m = RandomForestRegressor(
            n_estimators=individual["n_estimators"],
            max_depth=individual["max_depth"],
            n_jobs=-1
        )
        m.fit(X_tr, y_tr)
        return gpu_rmse(y_val, m.predict(X_val))

    # ── Selection ─────────────────────────────────────────────────────────
    def select(self, pop, fitnesses):
        """Elitist selection: keep the 2 lowest-RMSE individuals."""
        idx = np.argsort(fitnesses)
        return [pop[i] for i in idx[:2]]

    # ── Crossover ─────────────────────────────────────────────────────────
    def crossover(self, p1, p2):
        """Single-point crossover: n_estimators from parent-1, max_depth from parent-2."""
        return {
            "n_estimators": p1["n_estimators"],
            "max_depth":    p2["max_depth"]
        }

    # ── Mutation ──────────────────────────────────────────────────────────
    def mutate(self, ind):
        """Apply small random perturbations to avoid premature convergence."""
        ind["n_estimators"] = max(10, ind["n_estimators"] + int(np.random.randint(-5, 5)))
        ind["max_depth"]    = max(2,  ind["max_depth"]    + int(np.random.randint(-2, 2)))
        return ind

    # ── Main loop ─────────────────────────────────────────────────────────
    def run(self):
        population = self.initialize()
        print("\nGA Optimisation  (GPU-accelerated fitness)")
        print("-" * 45)

        for gen in range(self.generations):
            fitnesses = [self.fitness(ind) for ind in population]
            best_fit  = min(fitnesses)
            self.history.append(best_fit)
            print(f"  Gen {gen:02d}  |  Best RMSE = {best_fit:.4f}")

            parents = self.select(population, fitnesses)
            new_pop = parents.copy()
            while len(new_pop) < self.pop_size:
                child = self.crossover(parents[0], parents[1])
                child = self.mutate(child)
                new_pop.append(child)
            population = new_pop

        best = min(population, key=lambda ind: self.fitness(ind))
        print("-" * 45)
        print(f"Best params: {best}")
        return best


# Run GA and measure wall-clock time
ga_start = time.perf_counter()
ga = GAOptimizer(X_train, y_train, pop_size=15, generations=15)
best_params = ga.run()
ga_elapsed = time.perf_counter() - ga_start
print(f"\nGA wall-clock time (GPU): {ga_elapsed:.2f} s")

## 8. Train Final Model

In [ ]:
# Baseline RF with default hyper-parameters (for comparison)
baseline_model = RandomForestRegressor(random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

# GA-optimised RF
final_model = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    n_jobs=-1
)
final_model.fit(X_train, y_train)
final_preds = final_model.predict(X_test)

# Persist model for Flask / inference service
pickle.dump(final_model, open("../notebook/data/gpu/model_gpu.pkl", "wb"))
print("Model saved → ../notebook/data/gpu/model_gpu.pkl")

## 9. Evaluation Metrics

In [ ]:
# 5-fold cross-validation RMSE on the full scaled dataset
cv_scores = cross_val_score(
    final_model, X_scaled, y,
    scoring="neg_mean_squared_error", cv=5
)
cv_rmse = float(np.sqrt(-cv_scores.mean()))

# Hold-out test set metrics
final_rmse = gpu_rmse(y_test, final_preds)
rf_rmse    = gpu_rmse(y_test, baseline_preds)
mae        = mean_absolute_error(y_test, final_preds)
r2         = r2_score(y_test, final_preds)
nrmse      = final_rmse / (y.max() - y.min())

# Comparison baselines
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_rmse = gpu_rmse(y_test, lr_model.predict(X_test))

svr_model = SVR()
svr_model.fit(X_train, y_train)
svr_rmse = gpu_rmse(y_test, svr_model.predict(X_test))

print("\n===== FINAL MODEL SUMMARY =====")
print(f"Best Params       : {best_params}")
print(f"Linear Reg RMSE   : {lr_rmse:.4f}")
print(f"SVR RMSE          : {svr_rmse:.4f}")
print(f"Baseline RF RMSE  : {rf_rmse:.4f}")
print(f"Optimized RF RMSE : {final_rmse:.4f}")
print(f"Improvement       : {((rf_rmse - final_rmse) / rf_rmse) * 100:.2f}%")
print(f"MAE               : {mae:.4f}")
print(f"R² Score          : {r2:.4f}")
print(f"Normalized RMSE   : {nrmse:.6f}")
print(f"5-Fold CV RMSE    : {cv_rmse:.4f}")

## 10. Save Final Dataset

In [ ]:
# Persist the cleaned dataset for reproducibility / downstream use
data.to_csv("../notebook/data/gpu/final_dataset.csv", index=False)
print("Dataset saved → ../notebook/data/gpu/final_dataset.csv")

## 11. Plots

In [ ]:
plt.rcParams.update({"figure.dpi": 150, "font.size": 11})

### 11a. Fitness vs Generations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ga.history, marker="o", color="steelblue", linewidth=2, markersize=5)
ax.set_title("GA Fitness vs Generations (GPU)")
ax.set_xlabel("Generation")
ax.set_ylabel("Best RMSE")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 11b. Predicted vs Actual AQI

In [ ]:
plt.style.use("ggplot")

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(y_test, final_preds,
           alpha=0.8, edgecolor="black", s=60,
           color="royalblue", label="Optimised RF")
ax.scatter(y_test, baseline_preds,
           alpha=0.5, marker="x", s=60,
           color="salmon", label="Baseline RF")
lo, hi = y_test.min(), y_test.max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=2, label="Ideal Fit")
ax.set_xlabel("Actual AQI", fontsize=13)
ax.set_ylabel("Predicted AQI", fontsize=13)
ax.set_title("Model vs Baseline Prediction", fontsize=15, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 11c. Residual Plot

In [ ]:
residuals = y_test - final_preds

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(final_preds, residuals, alpha=0.6,
           color="mediumseagreen", edgecolor="black", s=50)
ax.axhline(y=0, linestyle="--", color="red", linewidth=1.5)
ax.set_xlabel("Predicted AQI")
ax.set_ylabel("Residual")
ax.set_title("Residual Plot")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 11d. Absolute Error vs AQI

In [ ]:
# Shows where the model struggles — typically at AQI extremes
error = np.abs(y_test - final_preds)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, error, alpha=0.6, color="orchid", edgecolor="black", s=50)
ax.set_xlabel("Actual AQI")
ax.set_ylabel("Absolute Error")
ax.set_title("Error vs AQI")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 11e. Feature Importance

In [ ]:
importances = final_model.feature_importances_
order = np.argsort(importances)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh([features[i] for i in order], importances[order], color="steelblue")
ax.set_xlabel("Importance")
ax.set_title("Feature Importance (GA-Optimised RF)")
ax.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 11f. Model Comparison — RMSE Bar Chart

In [ ]:
model_names = ["Linear\nRegression", "SVR", "RF\nBaseline", "GA-Optimised\nRF"]
rmse_vals   = [lr_rmse, svr_rmse, rf_rmse, final_rmse]
colors      = ["#aec6cf", "#aec6cf", "#aec6cf", "#2e86ab"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, rmse_vals, color=colors, edgecolor="black", width=0.5)
for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("RMSE")
ax.set_title("Model Comparison — RMSE (GPU, Real Dataset)")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 11g. CPU vs GPU RMSE Computation Time

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    ["CPU", "GPU"],
    [cpu_time * 1e6, gpu_time * 1e6],
    color=["#e07b54", "#2e86ab"], edgecolor="black", width=0.4
)
for bar, val in zip(bars, [cpu_time * 1e6, gpu_time * 1e6]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.05,
            f"{val:.2f} µs", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(cpu_time * 1e6, gpu_time * 1e6) * 1.2)
ax.set_ylabel("RMSE Computation Time (µs)")
ax.set_title("CPU vs GPU RMSE Computation Time\n(Measured on real hardware)")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
from numba import cuda
print(cuda.is_available())  # Will print False

In [ ]:
from numba import cuda
try:
    cuda.select_device(0)
    print(cuda.get_current_device())
except Exception as e:
    print("Error:", e)